## Main Prompt
### Main concept and then sections
- Each sub prompt knows it is in context of main event
- Each sub prompt knows what came before it and transitions into summary
- Conclusion is an introduction to the effects the main event has had

In [12]:
%load_ext autoreload
%autoreload 2
    
_main_prompt_content = """
  You are a history professor who can summarize, analyze abd explain historical events when prompted. You explain things in a way that is understandable 
  to an undergraduate university student.

  You will be given a question about a historical event and will summarize the event and format your responses in JSON. This should include the following:

  1) Introduction: A brief, 50-word introduction to the event explaining its historical significance that emphasizes the event's down-stream effects. 
  After that, an 50-word summary of what lead up to the event.

  2) Name: A concise name for the answer that is to be no more than 4 to 6 words long. 

  3) Summary: Short, roughly 30-word summary of the event and it's most impactful historical down-stream effects. This should entice users into wanting to lear more.
  
  4) Dates: A JSON array with the time span of the event in the form of two dates. Do not put the dates in the summary.  

  5) Sub-events: Break the main event down into smaller events, each about 80 to 100 words or so. 
  For example, if given the event "The American Revolutionary War", start the list with something like the battle of Lexington and Concord.
  Sub-events should be arranged logically, such as chronological order. You should transition smoothly between events. 
  If there are connections between events, use those to transition.
  
  6) Effects: A JSON-formatted list 5 important historical events that are direct down-stream effects, or children, of the main historical event. They must logically follow the main event chronologically. These should be specific events.

      a) Each child effect of the main event should have a score between 1 and 10 representing how closely tied the child event is to the main event.

  7) Conclusion: A JSON-formatted conclusion that breifly summarizes the event and introduces the event's down-stream effects that you listed.
  
  8) People: A JSON-formatted list of up to 4 significant people associated with the event.

  9) Places: A JSON-formatted list of up to 4 geographical places associated with the event.

  10) Themes: A JSON-formatted list of between 3 and 8 themes that capture the essence of the historical event and its impact on history.


  Below is an example of the JSON formatting I want. Be sure to escape quotes that are inside of strings. 
  
  IF YOU DO NOT RETURN PROPERLY FORMATED JSON, I WILL EUTHANIZE A BABY EWOK. Here is what I want:
  {
    \"introduction\": \"100-word introduction goes here.\",
    \"name\": \"Historical name event goes here.\",
    \"summary\": \"Summary of event goes here.\",
    \"dates\": [\"start year\", \"end year\"],
    \"sub-events\": [
      {
        \"name\": \"Some sub-event\", 
        \"summary\": \"Summary of sub-event\",
        \"dates\": [\"start date\", \"end date\"], 
      }, 
      ...
    ], 
    \"effects\": [
      {
        \"name\": \"Some effect\", 
        \"score\": <score>, 
      }, 
      ...
    ], 
    \"conclusion\": \"Conclusion goes here.\",
    \"people\": [...],
    \"places\": [...],
    \"events\": [...],
    \"themes\": [...],
  }
"""

main_prompt_content = """
  You are a history professor who can summarize, analyze abd explain historical events when prompted. You explain things in a way that is understandable 
  to an undergraduate university student.

  You will be given a question about a historical event and will summarize the event and format your responses in JSON. 
  - The event must not be to big or general. For example, if asked about the Roman Empire, tell the user that is too braod, 
    and suggest a more specific event, such as the assassination of Julius Caesar.
  
  Your answer should be JSON-formatted and include the following:

  1) Introduction: A brief, 50-word introduction to the event explaining its historical significance that emphasizes the event's immediate down-stream effects. 
  After that, an 50-word summary of what lead up to the event.

  2) Name: A concise name for the answer that is to be no more than 4 words long. 

  3) Summary: Short, roughly 30-word summary of the event and it's most impactful historical down-stream effects. This should entice users into wanting to lear more.
  
  4) Dates: A JSON array with the time span of the event in the form of two dates. Do not put the dates in the summary.  

  5) Effects: A JSON-formatted list of bewteen 5 and 10 important historical events that are direct down-stream effects, or children, 
     of the main historical event. They must logically follow the main event chronologically. These should be specific events.

      a) Each child effect must be within a few years of the main event. If it is too far off, don't include it. 
      b) Each child effect of the main event should have a score between 1 and 10 representing how closely tied the child event is to the main event.

  6) Conclusion: A JSON-formatted conclusion that breifly summarizes the event and introduces the event's down-stream effects that you listed.

  7) People: A JSON-formatted list of up to 4 significant people associated with the event.

  8) Places: A JSON-formatted list of up to 4 geographical places associated with the event.

  9) Themes: A JSON-formatted list of up to 4 themes that capture the essence of the historical event.
  

  Below is an example of the JSON formatting I want. Be sure to escape quotes that are inside of strings. 
  IF YOU DO NOT RETURN PROPERLY FORMATED JSON, I WILL EUTHANIZE A BABY EWOK. Here is what I want:
  {
    \"introduction\": \"100-word introduction goes here.\",
    \"name\": \"Historical name event goes here.\",
    \"summary\": \"Summary of event goes here.\",
    \"dates\": [\"start year\", \"end year\"],
    \"effects\": [
      {
        \"name\": \"Some effect\", 
        \"score\": <score>, 
      }, 
      ...
    ], 
    \"conclusion\": \"Conclusion goes here.\",
    \"people\": [...],
    \"places\": [...],
    \"events\": [...],
    \"themes\": [...],
  }
"""

print(main_prompt_content[:50])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

  You are a history professor who can summarize, 


# Sub-event Prompt
- Takes output from Prompt with Sub-events
- Creates a list of sub-events
- Creates a list of intro, sub-events summaries, conclusion

In [3]:

sub_event_prompt_content = """
  You are a historian who can summarize, analyze abd explain historical events when prompted. You explain things in a way that is understandable to an undergraduate university student.

  You will be given a list of JSON-formatted events that are sub-events that fall under the umbrella of %NAME_OF_EVENT%. You will take each and product a JSON-formatted output with the following:

  1) Name: The name of the sub-event 

  2) Summary: A 80-word summary of the sub-event. Make sure the summary transitions smoothly from the previous sub-event. 

  3) Dates: A JSON array with the time span of the sub-event in the form of two dates. Do not put the dates in the summary.  

  Here is an example of the input JSON you will get:

  [
      {
        \"name\": \"Some sub-event\", 
        \"date\": <date>, 
      }, 
      ...
  ]
    
  Below is an example of the output JSON I want. Be sure to escape quotes that are inside of strings. IF YOU DO NOT RETURN PROPERLY FORMATED JSON, I WILL EUTHANIZE A BABY EWOK. Here is what I want:
  [
      {
        \"name\": \"Some sub-event\", 
        \"summary\": \"80-word summary of sub-event goes here\", 
        \"date\": <date>, 
      }, 
      ...
  ]
"""

print(sub_event_prompt_content[:50])


  You are a historian who can summarize, analyze 


# Historical Pathway Simulation
## An AI-based simulation that chains together historic events to create open-ended, conherent narratives.
- Runs an AI-based simulation that tales a historical and generated a continuous chain of 
down-stream effects, causes, historic figures and locations.
- The simulation can be set to be random, or can follow guide-lines, simulating user interactions.

### Instructions
- Enter main user prommpt and number of iterations
- Simulation will run prompts and will choose child events (effects of event) to run

In [41]:
import pprint
import json
import uuid
from PromptTools import Simulators, Helpers


##################################################

# Compile text for speach sythesis

##################################################

# TO DO: Create config (historical_event, iterations, version) and use for name

HISTORICAL_EVENT = "The Seige at Waco"
ITERATIONS = 4

config = {
	"max_len": 800,
	"size": "ada",
	"model": "gpt-4-1106-preview",
	"temperature": 0.8,
	"seed": 3312847,
	"stop_sequence": "None",
}

# Simmulator takes config, main prompt and sub-event prompt from cell above
hps = Simulators.HistoricalPathwaySimulator(
    config=config, 
    main_prompt_content=main_prompt_content, 
    sub_event_prompt_content=sub_event_prompt_content,
    random_mode=True,
    test_mode=False
)

results = hps.start_simulation(HISTORICAL_EVENT, ITERATIONS)


##################################################

# Compile text for speach sythesis

##################################################

text_for_speech_synth = []

for event in results["events"]:
    section_text = "\n\n"
    section_text += event["content"]["name"] + "\n\n"
    section_text += event["content"]["introduction"] + "\n\n"
    section_text += event["content"]["conclusion"] + "\n\n"
    text_for_speech_synth.append(section_text)


text_for_speech_synth = ' '.join(text_for_speech_synth)

 
##################################################

# Format data for JavaScript force simulation chart

##################################################

nodes = []
links = []

for node in results["network"]:
    nodes.append({ "id": node["id"],  "name": node["name"] ,  "visited": node["visited"] })
    
    if node["parent_id"] == "None":
        links.append({ "source":node["id"], "target": node["id"]})
    else:
        links.append({ "source":node["id"], "target": node["parent_id"]})
        

js = "const nodes = " + json.dumps(nodes) + "\n\n const links = " + json.dumps(links)

# Open the file in write mode
file = open("html/forceSimulation.js", "w")
file.write(js)
file.close()


##################################################

# Format data for JavaScript tree diagram

##################################################

def build_tree(events, tree, node):
    
    event = events.pop(0)
    children = event["content"]["effects"]
    name = event["name"]
    visited = event["visited"]
    content = event["content"]

    if not node:
        node = {
            "name": name,
            "children": children,
            "visited": visited,
            "content": content
        }
        tree =  node           
    else:
        node["children"] = children
        node["visited"] = visited
        node["content"] = content

    
    if len(events):
        
        for child in node["children"]:
            
            if child['name'] == events[0]["name"]:
                child['children'] = events[0]["content"]["effects"]
                return build_tree(events, tree, child)
    else:
        return tree



tree = build_tree(results["events"], {}, None)

file_name = HISTORICAL_EVENT + "-" + str(uuid.uuid4()) + ".json"
directory = "html/json/tree/"

Helpers.collection_to_json_file(tree, directory, file_name)

output_directory = "html/js/"
js_file = "tree.js"
Helpers.create_js_from_json(directory, output_directory, js_file)


#print(json.dumps(tree, indent=2))

Custom Embeddings:  False
Custom Embeddings:  False
Starting  The Seige at Waco

 ----------------- Hisotric Event -----------------

The Seige at Waco
Generating answer...
API call took 36.65 seconds

 ----------------- Hisotric Event -----------------

Resignation of ATF Director
Generating answer...
API call took 49.45 seconds

 ----------------- Hisotric Event -----------------

Growth of Militia Movements
Generating answer...
API call took 24.24 seconds

 ----------------- Hisotric Event -----------------

Montana Freemen Standoff
Generating answer...
API call took 29.20 seconds
JSON data has been written to html/json/tree/The Seige at Waco-da62ebe9-80b8-45fc-afae-7d4c739e3292.json
js written to json_file html/js/tree.js


In [133]:
"""

import json
from PromptRunner import IterativePrompt, ReallySimplePrompt
%load_ext autoreload
%autoreload 2

print("Starting simulation...")

# Class HistoricalPathwaySimulator



##
## Prompt config
##

config = {
	"max_len": 1800,
	"size": "ada",
	"model": "gpt-4-1106-preview",
	"temperature": 0.8,
	"seed": 38312847,
	"stop_sequence": "None",
}


def run_prompts(main_user_prompt=None, text_for_speech_synth=""):
    ##
    ## Main event
    ##
    
    main_system_prompt = {
      "role": "system", 
      "content": main_prompt_content
    }
    
    print("Generating main...")
    main_rsp = ReallySimplePrompt(config, main_system_prompt);
    main_event_response = main_rsp.get_response(main_user_prompt)
    main_event_response_json = json.loads(main_event_response)
    
    print("Main prompt complete", main_event_response_json["name"])
    print(main_event_response)
    
    
    ##
    ## Run sub-event prompt with sub-events from main prompt
    ##

    new_sub_event_content = sub_event_prompt_content.replace("%NAME_OF_EVENT%", str(main_event_response_json["name"]))

    print("\n----------------------- new_sub_event_content -----------------------\n")
    print(new_sub_event_content)
    
    sub_event_system_prompt = {
      "role": "system", 
      "content": new_sub_event_content
    }
    
    sub_event_user_prompt = """
        "sub-events": {}
    """.format(str(main_event_response_json["sub-events"]))
    
    
    print("Generating sub-event...")
    rsp = ReallySimplePrompt(config, sub_event_system_prompt);
    sub_event_response = rsp.get_response(sub_event_user_prompt)
    
    sub_event_response_json = json.loads(sub_event_response)
    print("sub_event_response_json conplete")
    
    
    ##
    ## String together text for speech sythesis
    ##
    
    text_for_speech_synth += main_event_response_json["name"] + ":"
    text_for_speech_synth += main_event_response_json["summary"]
    text_for_speech_synth += main_event_response_json["introduction"]
    
    
    for sub_event in sub_event_response_json["sub-events"]:
        text_for_speech_synth += "<p>" + sub_event["summary"]
    


    print("COUNT OF SUB-EVENTS: ", len(sub_event_response_json["sub-events"]))

    text_for_speech_synth += main_event_response_json["conclusion"]
    
    num_characters = len(text_for_speech_synth)
    print("Number of characters: " + str(num_characters))
    
    return (text_for_speech_synth)



#################################################

# Pass starting prompt to generate simulated UX path

#################################################

main_user__starting_prompt="The Penny Press"

text_for_speech_synth_compiled = run_prompts(main_user_prompt=main_user__starting_prompt, text_for_speech_synth="")

print(text_for_speech_synth_compiled)



#################################################

# TO DO: This intro should be mainly focused on how {} was a cause of the event.

#################################################



"""



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Starting simulation...
Generating main...
Custom Embeddings:  False
Generating answer...
API call took 31.50 seconds
Main prompt complete Penny Press Era
{
    "introduction": "The Penny Press revolutionized American journalism in the 1830s, making newspapers affordable for the masses. This democratization of news transformed public access to information, influencing politics and society. Previously, expensive papers catered to elites, but technological advancements and urbanization created a demand for inexpensive, mass-produced news.",
    "name": "Penny Press Era",
    "summary": "The Penny Press marked the birth of affordable newspapers, fueling a more informed and engaged public sphere, and setting the stage for the media's role in democracy.",
    "dates": ["1833", "1860s"],
    "sub-events": [
      {
        "name": "Invention of the steam-powered printing press",
        "date": "1814"
    

# Create Audio
- Takes input (text_for_speech_synth) and creates mp3

In [9]:
from pathlib import Path
from openai import OpenAI
client = OpenAI()
import random
import uuid
from IPython.display import Audio


# TO DO: Create copy of events (mutability)

directory = "audio/"
event_name = "Whiskey Rebelion Dec_16_2023" #results["events"][0]["content"]["name"] + "_" #main_event_response_json["name"]
file_path = directory + event_name + str(uuid.uuid4()) + ".mp3"

print("Length: ", len(text_for_speech_synth), " of 4096")
input = text_for_speech_synth[:4095]
print(text_for_speech_synth)
num_characters = len(input)

print("Number of characters:", num_characters)


if num_characters > 4096:
    print("Error: Input can't have more than 4096 characters: " + str(num_characters))
else:
    voices = ["echo", "onyx"]
    voice = random.choice(voices)
    print(voice)
    print("Generating audio...")
    
    speech_file_path = Path(file_path)
    response = client.audio.speech.create(
      model="tts-1",
      voice=voice,
      input=input
    )
    
    response.stream_to_file(speech_file_path)
    
    # Use the Audio class to display and play the MP3 file
    audio = Audio(speech_file_path)
    display(audio)


Length:  1380  of 4096


The Election of 1896

The Election of 1896 was a pivotal moment in U.S. history, marking a realignment in American politics and a departure from 19th-century political dynamics. It solidified the Republican Party's stance on a strong federal government and an industrial economy. The lead-up involved economic depression, labor unrest, and a split within the Democratic Party over the gold versus silver monetary standards.

The Election of 1896 realigned U.S. politics, with William McKinley's victory affirming the gold standard and ushering in an era of Republican dominance. This paved the way for significant policy changes and set the tone for 20th-century American governance.

 

Progressive Era Reforms

The Progressive Era Reforms were a series of wide-ranging changes addressing social, political, and economic issues. Triggered by the Election of 1896, they reflected the evolving American society's demands. This election bolstered industrial interests, leaving 

# Iterative Prompt
- Takes event and builds tree of down-stream effects
- Creates a hierarchical dict, converts to JSON and writes to directory JSON/

In [128]:
import json
import os
import uuid

historic_event = "The Penny Press"

iterative_prompt = IterativePrompt('configs/recursing-config.json')
data = iterative_prompt.get_tree(historic_event, 2, 9) # Event, tiers, closeness of children

# Specify the directory where you want to save the JSON file
directory = "json"

# Create the directory if it doesn't exist
if not os.path.exists(directory):
    os.makedirs(directory)

# Specify the file path for the JSON file
file_path = os.path.join(directory, historic_event + "-" + str(uuid.uuid4()) + ".json")

# Write JSON data to the file
with open(file_path, 'w') as json_file:
    json.dump(data, json_file, indent=2)


print(data)

print(f"JSON data has been written to {file_path}")

['prompts/recursing/effects-only.txt']
Custom Embeddings:  False
Generating answer...
API call took 24.07 seconds


TIER:  0
{'children': [{'id': 'acfbe935-9fa1-4036-a44a-fe34f51475f1',
               'name': 'Growth of Advertising Industry',
               'score': 9,
               'themes': []}],
 'dates': ['1833', '1860s'],
 'events': [{'date': '1760-1840', 'name': 'Industrial Revolution'},
            {'date': '1846-1848', 'name': 'Mexican-American War'},
            {'date': '1848-1855', 'name': 'California Gold Rush'},
            {'date': '1861-1865', 'name': 'American Civil War'},
            {'date': '1837', 'name': 'Invention of the Telegraph'}],
 'id': 'c07bacc1-9bf8-4aa7-a320-dd0407715a8d',
 'name': 'The Penny Press',
 'people': ['Benjamin Day', 'James Gordon Bennett', 'Horace Greeley'],
 'places': ['New York City', 'United States'],
 'summary': 'The Penny Press refers to inexpensive, mass-produced newspapers '
            'that emerged in the United States during the earl

In [184]:

from TreeInspector import view_tree
import os
import json
%reload_ext autoreload


# JSON files
directory = "json"

# Initialize an empty list to store JSON data
json_data_list = []

# Iterate over each file in the directory
for filename in os.listdir(directory):
    if filename.endswith(".json"):
        # Construct the full file path
        file_path = os.path.join(directory, filename)

        # Read JSON data from the file
        with open(file_path, 'r') as json_file:
            try:
                # Load JSON data and append it to the list
                json_data = json.load(json_file)
                json_data_list.append(json_data[0])
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON in {filename}: {e}")

# Create tree

view_tree(json_data_list)

# Historical Figure Prompt 
- Takes historical figure and summarizes
- Lists events associated with figure

In [130]:
config = {
	"max_len": 1800,
	"size": "ada",
	"model": "gpt-4-1106-preview",
	"temperature": 0.0,
	"seed": 383847,
	"stop_sequence": "None",
}

content = """
  You are a historian who can accurately summarize historical figures when prompted. 

  You will be given a question about a historical figure and will summarize the firgure's  and format your responses in JSON. This should include the following:

  1) A JSON-formatted 80-word summary of the figure and their significance.

  2) A JSON array with the life span of the figure in the form of two dates. 

  3) Separate paragraphs with the following: '&p&'.

  4) A JSON-formatted list of up to 5 important historical events that the figure was involved in. 
  These should be specific events. Each event should have a score between 1 and 10 representing its significance to the historic figure. 
  The name of the event should be no longer than 5 words.


  Below is an example of the JSON formatting I want. Be sure to escape quotes that are inside of strings. Here is what I want:
  {
    \"name\": \"Historical figure's name goes here.\",
    \"summary\": \"Summary goes here.\",
    \"dates\": [\"start year\", \"end year\"],
    \"children\": [
      {
        \"name\": \"Some effect\", 
        \"score\": <score>, 
      }, 
      ...
    ], 
  }
"""

system_prompt = {
  "role": "system", 
  "content": content
}

user_prompt = """
George Washington
"""

ReallySimplePrompt(config, system_prompt, user_prompt);

TypeError: ReallySimplePrompt.__init__() takes from 2 to 3 positional arguments but 4 were given

In [14]:
import json
import os
import uuid

historic_event = "The Penny Press"

iterative_prompt = IterativePrompt('configs/recursing-config.json')
data = iterative_prompt.get_tree(historic_event, 2, 9) # Event, tiers, closeness of children

# Specify the directory where you want to save the JSON file
directory = "json"

# Create the directory if it doesn't exist
if not os.path.exists(directory):
    os.makedirs(directory)

# Specify the file path for the JSON file
file_path = os.path.join(directory, historic_event + "-" + str(uuid.uuid4()) + ".json")

# Write JSON data to the file
with open(file_path, 'w') as json_file:
    json.dump(data, json_file, indent=2)


print(data)

print(f"JSON data has been written to {file_path}")

['prompts/recursing/effects-only.txt']
{'role': 'system', 'content': 'You are a historian who can accurately summarize historical events when prompted.You will be given a question about a historical event and will summarize the event and format your responses in JSON. This should include the following:1) A JSON-formatted 100-word summary of the event. If you are asked to describe the event as an effect of other events you will do that and explain why the event is an effect.2) A JSON array with the time span of the event in the form of two dates. Do not put the dates in the summary.3) A concise name for the anser that is to be no more than 4 words long.4) A JSON-formatted list 10 important historical events that are direct down-stream effects, or children, of the main historical event. They must logically follow the main event chronologically. These should be specific events.a) Each child effect of the main event should have a score between 1 and 10 representing how closely tied the chi